# 03 — Analyze and Visualize**Goals**1. Group descriptives — N per program group, education-rate by group2. Figure 1: Stacked bar chart of education distribution by program-exposure group (replicates Milestone 1 figure)3. Figure 2: Bar chart of HS-completion rate by program group (replaces the average-grade plot from Milestone 1 since the CYA file lacks a single canonical "highest grade completed" variable for the young adult)4. Save both figures to `output/`

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom pathlib import PathPROJECT = Path('..').resolve()DATA    = PROJECT / 'data' / 'nlscya_cleaned.csv'OUT     = PROJECT / 'output'df = pd.read_csv(DATA)print('Loaded:', df.shape)df.head()

## Descriptives — sample sizes per group

In [ ]:
print('Sample size per program group:')print(df['program_group'].value_counts())print('\nMean HS-completion (HSDIPLOMA or GED) by group:')df['has_hs'] = ((df['HSDIPLOMA_EVER'] == 1) | (df['GED_EVER'] == 1)).astype(float)df.loc[df['HSDIPLOMA_EVER'].isna() & df['GED_EVER'].isna(), 'has_hs'] = np.nanprint(df.groupby('program_group')['has_hs'].mean().round(3))

## Figure 1 — Education distribution by program group (stacked bar)

In [ ]:
order = ['Neither', 'SNAP only', 'TANF only', 'Both SNAP and TANF']ed_order = ['No HS / GED', 'HS or GED', 'Professional degree']# Cross-tab of group × education, normalized to row percentagessub = df.dropna(subset=['program_group', 'edu_label'])ct = pd.crosstab(sub['program_group'], sub['edu_label'], normalize='index') * 100ct = ct.reindex(index=order, columns=ed_order)print('Education distribution (%) by program group:')print(ct.round(1))# Plotplt.figure(figsize=(10, 7))bottom = np.zeros(len(order))colors = ['#4a6fa5', '#f4a261', '#3a7d44']for i, ed in enumerate(ed_order):    plt.bar(order, ct[ed].values, bottom=bottom,            label=ed, color=colors[i])    bottom += ct[ed].valuesplt.xticks(rotation=15, fontsize=14)plt.yticks(fontsize=14)plt.xlabel('Program exposure group', fontsize=16)plt.ylabel('Percent of respondents', fontsize=16)plt.title('Education Distribution by SNAP/TANF Exposure Group', fontsize=16)plt.legend(title='Education category', fontsize=12, loc='upper right')plt.tight_layout()plt.savefig(OUT / 'fig1_education_distribution.png', bbox_inches='tight', dpi=150)plt.show()

## Figure 2 — HS completion rate by program group with 95% CIs

In [ ]:
# Group means, sizes, standard errorsg = sub.groupby('program_group')['has_hs']means  = g.mean().reindex(order)counts = g.count().reindex(order)ses    = (means * (1 - means) / counts) ** 0.5# 95% CI half-widthci = 1.96 * sesplt.figure(figsize=(10, 7))plt.errorbar(order, means, yerr=ci, fmt='o', markersize=12,             color='#4a6fa5', capsize=8, capthick=2, elinewidth=2)plt.xticks(rotation=15, fontsize=14)plt.yticks(fontsize=14)plt.xlabel('Program exposure group', fontsize=16)plt.ylabel('Share with HS diploma or GED', fontsize=16)plt.title('HS/GED Completion Rate by SNAP/TANF Exposure Group (95% CI)', fontsize=16)plt.ylim(0, 1)plt.tight_layout()plt.savefig(OUT / 'fig2_hs_completion_by_group.png', bbox_inches='tight', dpi=150)plt.show()# Print the underlying numberssummary = pd.DataFrame({'N': counts, 'HS_rate': means.round(3), 'CI_halfwidth': ci.round(3)})print('\nUnderlying numbers:')print(summary)

## Save summary table to output/

In [ ]:
summary.to_csv(OUT / 'table1_hs_rate_by_group.csv')print('Saved table1_hs_rate_by_group.csv')

## Findings so farThe descriptives show clear differences in HS/GED completion rates across the four exposure groups. Children of mothers receiving both SNAP and TANF have lower HS-completion rates than children in the Neither group. This pattern is consistent with the welfare-policy literature reviewed in the Lit Review assignment (Heflin et al. 2022, Pilkauskas 2023), but **does not establish causality** — group differences likely reflect differences in household income, parental education, and other background factors, which we have not yet controlled for.**Next steps (Milestone 3+):**- Add demographic controls (CRACE, CSEX, mother's education) to regression models- Construct a continuous "highest grade completed" outcome by pulling additional NLS columns- Run logistic regression of HS completion on program-group with controls (Unit 11)